# 02 · Exploratory analysis and speaker-independent splits

**Input:** `manifest.csv` from notebook 01 (attach its output on Kaggle).
**Outputs:** `splits.csv` (kept utterances + `split`, `split_random`, `prompt_in_train*`),
`split_report.json`, figures in `figures/`.

Every chart below answers a specific question that affects a modelling or evaluation decision.

In [ ]:
# ── Setup: locate the project source (src/dholuo_asr) and the output folder ──────────
import os, sys, subprocess, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# After pushing the repo to GitHub, put its URL here (Kaggle: Settings → Internet ON).
# Alternatively upload the repo as a private Kaggle dataset and attach it as an input.
REPO_URL = os.environ.get("DHOLUO_REPO_URL", "https://github.com/YOUR-USERNAME/dholuo-asr.git")

def _find_src(max_depth=4):
    for root in [Path.cwd(), Path.cwd().parent, Path("/kaggle/working"), Path("/kaggle/input")]:
        if not root.exists():
            continue
        base = len(root.parts)
        for dirpath, dirnames, filenames in os.walk(root):
            if len(Path(dirpath).parts) - base >= max_depth or len(filenames) > 500:
                dirnames[:] = []          # don't descend into audio folders
            if Path(dirpath).name == "dholuo_asr" and "__init__.py" in filenames:
                return Path(dirpath).parent
    return None

SRC = _find_src()
if SRC is None and "YOUR-USERNAME" not in REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/kaggle/working/dholuo-asr"], check=True)
    SRC = Path("/kaggle/working/dholuo-asr/src")
if SRC is None:
    raise RuntimeError("Project source not found: set REPO_URL (Internet ON) or attach the repo as a dataset.")
sys.path.insert(0, str(SRC))
REPO = SRC.parent
ON_KAGGLE = Path("/kaggle/working").exists()
OUT = Path(os.environ.get("DHOLUO_OUT", "/kaggle/working" if ON_KAGGLE else REPO / "outputs"))
OUT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOTS = [OUT, Path("/kaggle/input")]
print("source :", SRC)
print("outputs:", OUT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})
DIALECT_COLORS = {"Milambo": "#2a6f97", "Nyanduat": "#e07a5f"}
FIG = OUT / "figures"; FIG.mkdir(exist_ok=True)
def savefig(name):
    plt.tight_layout(); plt.savefig(FIG / f"{name}.png", bbox_inches="tight"); plt.show()

In [ ]:
from dholuo_asr.utils import find_artifact, dump_json
from collections import Counter
from dholuo_asr.text import normalise, minimal_normalise
SPLIT_SEED = 42
RATIOS = (0.70, 0.15, 0.15)          # train/val/test share of UTTERANCES, assigned by SPEAKER
man = pd.read_csv(find_artifact("manifest.csv", ARTIFACT_ROOTS), keep_default_na=False, na_values=[""], low_memory=False)
df = man[man["keep"].astype(str).str.lower() == "true"].copy()
df["text_raw"] = df["text_raw"].fillna("").astype(str)
df["text_norm"] = df["text_raw"].map(normalise)
df["text_minimal"] = df["text_raw"].map(minimal_normalise)
for c in ["speaker", "dialect", "gender", "age", "domain"]:
    df[c] = df[c].astype("string")
print(f"{len(df):,} kept utterances | {df['speaker'].nunique()} speakers | {df['duration_s'].sum()/3600:.2f} h")

### Q1. How balanced are the dialects, in speakers and in hours?
Determines whether a single pooled WER would be dominated by one dialect (Experiment D).

In [ ]:
g = df.groupby("dialect").agg(speakers=("speaker", "nunique"), utterances=("utt_id", "size"), hours=("duration_s", lambda s: s.sum() / 3600))
display(g.assign(**{"hours_share": g.hours / g.hours.sum()}).round(3))
fig, ax = plt.subplots(1, 3, figsize=(11, 3))
for a, col in zip(ax, ["speakers", "utterances", "hours"]):
    a.bar(g.index, g[col], color=[DIALECT_COLORS.get(i, "grey") for i in g.index]); a.set_title(col)
savefig("q1_dialect_balance")

### Q2. Is the corpus dominated by a few speakers?
If a handful of speakers contribute most audio, a random split would put the same voices in train and test, and per-speaker variance will dominate the test-set confidence interval.

In [ ]:
spk = df.groupby(["speaker", "dialect"]).agg(utts=("utt_id", "size"), minutes=("duration_s", lambda s: s.sum() / 60)).reset_index().sort_values("minutes", ascending=False)
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.bar(range(len(spk)), spk["minutes"], color=[DIALECT_COLORS.get(d, "grey") for d in spk["dialect"]])
ax.set_xlabel("speaker (sorted)"); ax.set_ylabel("minutes of audio"); ax.set_title("Audio per speaker (colour = dialect)")
savefig("q2_per_speaker")
top5 = spk["minutes"].head(5).sum() / spk["minutes"].sum()
print(f"top-5 speakers contribute {top5:.1%} of audio; median {spk['minutes'].median():.1f} min/speaker, min {spk['minutes'].min():.1f}, max {spk['minutes'].max():.1f}")

### Q3. Demographics: gender and age by dialect
Needed for the responsible-NLP discussion and to check the split does not, e.g., put all women of one dialect in test.

In [ ]:
for col in ["gender", "age"]:
    if df[col].notna().any():
        display(pd.crosstab(df.drop_duplicates("speaker")[col].fillna("unknown"), df.drop_duplicates("speaker")["dialect"].fillna("unknown"), margins=True).rename_axis(f"speakers by {col}"))

### Q4. How long are utterances? (sets memory limits and the 15 s training cap)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
dur_bins = np.linspace(0, df["duration_s"].quantile(0.999) * 1.05, 60)
for d, sub in df.groupby("dialect"):
    ax[0].hist(sub["duration_s"], bins=dur_bins, alpha=0.6, label=d, color=DIALECT_COLORS.get(d))
    ax[1].hist(sub["n_words"], bins=range(0, int(df["n_words"].max()) + 2), alpha=0.6, label=d, color=DIALECT_COLORS.get(d))
ax[0].axvline(15, ls="--", c="k", lw=1); ax[0].set_xlabel("seconds"); ax[0].set_title("Duration")
ax[1].set_xlabel("words"); ax[1].set_title("Words per utterance"); ax[0].legend()
savefig("q4_lengths")
print(df[["duration_s", "n_words"]].describe(percentiles=[.05, .5, .95, .99]).round(2).T)
print(f"utterances > 15 s (skipped in training only): {(df['duration_s'] > 15).sum():,}")

### Q5. What domains does the text cover?

In [ ]:
if df["domain"].notna().any():
    dom = pd.crosstab(df["domain"].fillna("unknown"), df["dialect"].fillna("unknown"))
    dom.plot.barh(stacked=True, figsize=(8, 0.35 * len(dom) + 1.5), color=[DIALECT_COLORS.get(c, "grey") for c in dom.columns])
    plt.title("Utterances per domain"); savefig("q5_domains")
else:
    print("No domain field: domain-level analysis not possible with this dataset.")

### Q6. How much do speakers share prompts? (the main leakage risk besides speaker identity)

In [ ]:
pp = df.groupby("text_norm")["speaker"].nunique()
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(pp, bins=range(1, int(pp.max()) + 2), color="#555")
ax.set_xlabel("number of speakers reading the same prompt"); ax.set_ylabel("prompts"); ax.set_yscale("log")
savefig("q6_prompt_sharing")
print(f"{(pp >= 2).mean():.1%} of prompts are read by >=2 speakers")

### Q7. How fast does the vocabulary grow? (expected out-of-vocabulary rate on unseen text)
A type–token curve that keeps rising means a word-level LM trained on this corpus will meet many unseen words; character-level CTC is the right output unit.

In [ ]:
toks = " ".join(df.sample(frac=1, random_state=0)["text_norm"]).split()
xs = np.unique(np.linspace(100, len(toks), 60).astype(int))
seen, types, k = set(), [], 0
for x in xs:
    seen.update(toks[k:x]); k = x; types.append(len(seen))
wc = Counter(toks)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(xs, types); ax[0].set_xlabel("tokens"); ax[0].set_ylabel("distinct words"); ax[0].set_title("Vocabulary growth")
r = np.arange(1, len(wc) + 1); ax[1].loglog(r, sorted(wc.values(), reverse=True)); ax[1].set_title("Rank–frequency (Zipf)")
savefig("q7_vocab")
print(f"{len(toks):,} tokens, {len(wc):,} types, {sum(1 for v in wc.values() if v == 1):,} hapax legomena")
print("most frequent:", wc.most_common(20))
chars = Counter("".join(df["text_norm"]).replace(" ", ""))
print("character inventory (normalised):", "".join(sorted(chars)))

## Speaker-independent split
Speakers are assigned whole to train/val/test, stratified by dialect, with the assignment chosen (seeded search) to match the target utterance shares overall and within each gender. **No speaker appears in more than one split.**

In [ ]:
from collections import Counter
from dholuo_asr.splits import speaker_split, random_split, leakage_report
assign = speaker_split(df, ratios=RATIOS, seed=SPLIT_SEED, stratify_col="dialect",
                       balance_cols=tuple(c for c in ["gender"] if df[c].notna().any()))
df["split"] = df["speaker"].map(assign)
assert df["split"].notna().all()
tab = df.groupby(["split", "dialect"]).agg(utts=("utt_id", "size"), speakers=("speaker", "nunique"), hours=("duration_s", lambda s: s.sum() / 3600)).round(2)
display(tab.unstack("dialect"))
if df["gender"].notna().any():
    display(pd.crosstab(df.drop_duplicates("speaker")["split"], df.drop_duplicates("speaker")["gender"].fillna("unknown")).rename_axis("speakers"))
rep = leakage_report(df, "split")
assert rep["speaker_overlap_train_test"] == 0 and rep["speaker_overlap_train_val"] == 0 and rep["speaker_overlap_val_test"] == 0
rep

### Prompt overlap between splits
Even with disjoint speakers, a test sentence may also be read by training speakers. Every test utterance gets a `prompt_in_train` flag; results are reported on the full test set **and** on the unseen-prompt subset, which is the honest measure of generalisation to new sentences.

In [ ]:
train_txt = set(df.loc[df.split == "train", "text_norm"])
df["prompt_in_train"] = df["text_norm"].isin(train_txt)
print(df.groupby("split")["prompt_in_train"].agg(["mean", "sum", "size"]).round(3))
test_unseen = df[(df.split == "test") & ~df.prompt_in_train]
print(f"unseen-prompt test subset: {len(test_unseen):,} utterances, {test_unseen['speaker'].nunique()} speakers, {test_unseen['duration_s'].sum()/3600:.2f} h")
if len(test_unseen) < 200:
    print("WARNING: the unseen-prompt subset is small; its confidence intervals will be wide.")

### Random row-level split — for Experiment F only
Same number of utterances per split as the official split, but rows assigned at random, so the same speakers (and prompts) appear in train and test. **Never used as an official result**; it exists to measure how much a leaky evaluation would flatter the model.

In [ ]:
sizes = df["split"].value_counts().to_dict()
df["split_random"] = random_split(df, sizes, seed=SPLIT_SEED)
rep_random = leakage_report(df, "split_random")
txt_r = set(df.loc[df.split_random == "train", "text_norm"])
df["prompt_in_train_random"] = df["text_norm"].isin(txt_r)
print({k: v for k, v in rep_random.items() if "overlap" in k or "frac" in k})

### Speaker- AND prompt-disjoint variant (`split_strict`)
If most test prompts were also read by training speakers, the official test set mainly measures recognition of
*known sentences by new voices*. `split_strict` keeps the same val/test speakers but **removes from training every
utterance whose prompt occurs in val or test**. Training on it (`xlsr_full_prompt_disjoint`) measures generalisation
to new voices **and** new sentences, at the cost of training data. The recommendation printed below is based on the
measured overlap.

In [ ]:
heldout_txt = set(df.loc[df.split.isin(["val", "test"]), "text_norm"])
df["split_strict"] = df["split"].where(~((df.split == "train") & df.text_norm.isin(heldout_txt)), "excluded")
lost = (df.split_strict == "excluded").sum()
print(f"strict split removes {lost:,} training utterances ({lost / (df.split == 'train').sum():.1%} of train, "
      f"{df.loc[df.split_strict == 'excluded', 'duration_s'].sum()/3600:.2f} h)")
print(df.groupby("split_strict").agg(utts=("utt_id", "size"), speakers=("speaker", "nunique")))
frac = rep["test_frac_prompt_in_train"]
if frac > 0.5:
    print(f"RECOMMENDATION: {frac:.0%} of test utterances have a prompt seen in training -> run xlsr_full_prompt_disjoint "
          "and report it alongside the official split.")
elif frac > 0.1:
    print(f"{frac:.0%} prompt overlap: report the unseen-prompt subset; the strict run is optional.")
else:
    print(f"only {frac:.0%} prompt overlap: the official split already tests unseen sentences.")

### Code-switching flags (verified if a verified file is attached)

In [ ]:
try:
    ver = pd.read_csv(find_artifact("codeswitch_candidates_verified.csv", ARTIFACT_ROOTS))
    yes = set(ver.loc[ver["verified"].astype(str).str.lower().isin(["yes", "y", "1", "true"]), "token"])
    df["cs_candidate"] = df["text_norm"].map(lambda t: any(w in yes for w in t.split()))
    CS_VERIFIED = True
except FileNotFoundError:
    df["cs_candidate"] = df["cs_candidate"].astype(str).str.lower() == "true"
    CS_VERIFIED = False
print("code-switch flags verified by a speaker:", CS_VERIFIED, "| flagged utterances:", int(df["cs_candidate"].sum()))

In [ ]:
cols = ["utt_id", "audio_path", "duration_s", "text_raw", "text_norm", "speaker", "dialect", "gender", "age",
        "domain", "split", "split_random", "split_strict", "prompt_in_train", "prompt_in_train_random", "cs_candidate", "n_words",
        "rms_dbfs", "speech_frac", "flag_fast_speech", "flag_slow_speech", "has_tag", "has_digit"]
df[[c for c in cols if c in df]].to_csv(OUT / "splits.csv", index=False)
dump_json({"seed": SPLIT_SEED, "ratios": RATIOS, "speaker_split": rep, "random_split": rep_random,
           "cs_verified": CS_VERIFIED, "table": tab.reset_index().to_dict("records"),
           "speakers_per_split": df.groupby("split")["speaker"].nunique().to_dict()},
          OUT / "split_report.json")
print("wrote", OUT / "splits.csv")